# DCE-MRI — statistical analysis

Patient-level descriptive statistics, bootstrap confidence intervals, paired Wilcoxon tests, and Holm multiplicity correction.


In [ ]:
from __future__ import annotations
import json, warnings, zipfile
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/ROI_MRI_TRACKB_FINAL_STATISTICAL_ANALYSIS')
TABLES=WORK/'tables';FIGURES=WORK/'figures';LOGS=WORK/'logs'
for p in [WORK,TABLES,FIGURES,LOGS]:p.mkdir(parents=True,exist_ok=True)

N_BOOTSTRAP=5000; BOOTSTRAP_SEED=2026; ALPHA=0.05
MODELS=['unet','attention_unet','swin_tiny_unet']
LABEL={'unet':'U-Net','attention_unet':'Attention U-Net','swin_tiny_unet':'Swin-Tiny-U-Net'}
SPLITS=['validation','test','external']
METRICS=['patient_dice','patient_iou','patient_precision','patient_recall']
rng=np.random.default_rng(BOOTSTRAP_SEED)
print('Output:',WORK)

In [ ]:
def find(name,roots):
    out=[]
    for r in roots:
        if r.exists():out+=list(r.rglob(name))
    return sorted(set(out))
roots=[INPUT]
needed=['crop_level_metrics.csv','patient_level_metrics.csv','patient_metrics_seed_averaged.csv']
found={n:find(n,roots) for n in needed}
if not all(found.values()):
    zips=[p for p in INPUT.rglob('*.zip') if 'PATIENT_LEVEL_EVALUATION' in p.name.upper() or 'TRACKB_FINAL' in p.name.upper()]
    if not zips:raise FileNotFoundError('Add TRACKB_FINAL_PATIENT_LEVEL_EVALUATION_LIGHT.zip')
    dest=Path('/kaggle/working/notebook1_extracted');dest.mkdir(parents=True,exist_ok=True)
    for z in zips:
        with zipfile.ZipFile(z) as f:f.extractall(dest)
    roots.append(dest);found={n:find(n,roots) for n in needed}
if not all(found.values()):raise FileNotFoundError('Required CSV files not found after extraction.')
paths={n:max(v,key=lambda p:p.stat().st_size) for n,v in found.items()}
crop=pd.read_csv(paths['crop_level_metrics.csv']);patient=pd.read_csv(paths['patient_level_metrics.csv']);seedavg=pd.read_csv(paths['patient_metrics_seed_averaged.csv'])
print({k:str(v) for k,v in paths.items()});print('Rows:',len(crop),len(patient),len(seedavg))

In [ ]:
required={'patient_id','dataset','split','model','patient_dice','patient_iou','patient_precision','patient_recall'}
if required-set(seedavg.columns):raise ValueError(f'Missing columns: {required-set(seedavg.columns)}')
assert not seedavg.duplicated(['patient_id','dataset','split','model']).any()
coverage=seedavg.groupby(['patient_id','dataset','split']).model.nunique()
assert coverage.eq(3).all()
integrity={'crop_rows':len(crop),'patient_seed_rows':len(patient),'patient_seed_averaged_rows':len(seedavg),'complete_three_model_coverage':True}
(LOGS/'statistical_input_integrity.json').write_text(json.dumps(integrity,indent=2))
print(integrity)

In [ ]:
rows=[]
for (model,split),g in seedavg.groupby(['model','split']):
    for metric in METRICS:
        x=g[metric].dropna()
        rows.append({'model':model,'model_label':LABEL[model],'split':split,'metric':metric,'n_patients':len(x),'mean':x.mean(),'std':x.std(ddof=1),'median':x.median(),'q1':x.quantile(.25),'q3':x.quantile(.75),'min':x.min(),'max':x.max()})
descriptive=pd.DataFrame(rows);descriptive.to_csv(TABLES/'patient_level_descriptive_statistics.csv',index=False)
print(descriptive[descriptive.metric=='patient_dice'].to_string(index=False))

In [ ]:
def boot_mean(x,nboot=N_BOOTSTRAP):
    x=np.asarray(x,float);x=x[np.isfinite(x)];n=len(x);means=np.empty(nboot);chunk=500
    for a in range(0,nboot,chunk):
        b=min(a+chunk,nboot);idx=rng.integers(0,n,size=(b-a,n));means[a:b]=x[idx].mean(1)
    return x.mean(),x.std(ddof=1),np.quantile(means,.025),np.quantile(means,.975)

rows=[]
for (model,split),g in seedavg.groupby(['model','split']):
    for metric in METRICS:
        mean,std,lo,hi=boot_mean(g[metric])
        rows.append({'model':model,'model_label':LABEL[model],'split':split,'metric':metric,'n_patients':g[metric].notna().sum(),'mean':mean,'std':std,'ci95_lower':lo,'ci95_upper':hi,'bootstrap_unit':'patient','n_bootstrap':N_BOOTSTRAP})
ci=pd.DataFrame(rows);ci.to_csv(TABLES/'patient_level_bootstrap_ci.csv',index=False)
print(ci[ci.metric=='patient_dice'].to_string(index=False))

In [ ]:
def boot_diff(a,b,nboot=N_BOOTSTRAP):
    a,b=np.asarray(a,float),np.asarray(b,float);d=a-b;d=d[np.isfinite(d)];n=len(d);means=np.empty(nboot);chunk=500
    for x in range(0,nboot,chunk):
        y=min(x+chunk,nboot);idx=rng.integers(0,n,size=(y-x,n));means[x:y]=d[idx].mean(1)
    return d.mean(),np.median(d),np.quantile(means,.025),np.quantile(means,.975),n

def holm(p):
    p=np.asarray(p,float);m=len(p);order=np.argsort(p);adj=np.empty(m);running=0
    for rank,idx in enumerate(order):running=max(running,(m-rank)*p[idx]);adj[idx]=min(running,1)
    return adj

rows=[]
for split in SPLITS:
  df=seedavg[seedavg.split==split]
  for metric in METRICS:
    pivot=df.pivot_table(index=['patient_id','dataset'],columns='model',values=metric,aggfunc='first').dropna(subset=MODELS)
    for a,b in combinations(MODELS,2):
        mean,med,lo,hi,n=boot_diff(pivot[a],pivot[b])
        try:stat,p=wilcoxon(pivot[a],pivot[b],zero_method='wilcox')
        except ValueError:stat,p=np.nan,1.
        rows.append({'split':split,'metric':metric,'model_a':a,'model_b':b,'comparison':f'{LABEL[a]} - {LABEL[b]}','n_patients':n,'mean_difference_a_minus_b':mean,'median_difference_a_minus_b':med,'bootstrap_ci95_lower':lo,'bootstrap_ci95_upper':hi,'ci_excludes_zero':bool(lo>0 or hi<0),'wilcoxon_statistic':stat,'p_value':p})
comp=pd.DataFrame(rows);comp['p_value_holm']=np.nan
for _,idx in comp.groupby(['split','metric']).groups.items():comp.loc[idx,'p_value_holm']=holm(comp.loc[idx,'p_value'])
comp['significant_holm_0_05']=comp.p_value_holm<.05
comp.to_csv(TABLES/'paired_model_comparisons.csv',index=False)
print(comp[(comp.metric=='patient_dice') & ((comp.model_a=='swin_tiny_unet')|(comp.model_b=='swin_tiny_unet'))].to_string(index=False))

In [ ]:
rows=[]
for (dataset,split,model),g in seedavg.groupby(['dataset','split','model']):
    for metric in METRICS:
        mean,std,lo,hi=boot_mean(g[metric],min(N_BOOTSTRAP,3000))
        rows.append({'dataset':dataset,'split':split,'model':model,'model_label':LABEL[model],'metric':metric,'n_patients':g.patient_id.nunique(),'mean':mean,'std':std,'ci95_lower':lo,'ci95_upper':hi})
source=pd.DataFrame(rows);source.to_csv(TABLES/'results_by_source_dataset.csv',index=False)

sizes=seedavg.groupby(['patient_id','dataset','split'],as_index=False).total_lesion_pixels.mean()
sizes['lesion_size_quartile']=sizes.groupby('split').total_lesion_pixels.transform(lambda x:pd.qcut(x.rank(method='first'),4,labels=['Q1_smallest','Q2','Q3','Q4_largest']))
sub=seedavg.merge(sizes[['patient_id','dataset','split','lesion_size_quartile']],on=['patient_id','dataset','split'])
sub['border_group']=np.where(sub.any_crop_touches_border.astype(bool),'touches_border','does_not_touch_border')
size_res=sub.groupby(['split','lesion_size_quartile','model'],observed=True).agg(n_patients=('patient_id','nunique'),patient_dice_mean=('patient_dice','mean'),patient_dice_std=('patient_dice','std'),patient_recall_mean=('patient_recall','mean'),patient_precision_mean=('patient_precision','mean')).reset_index()
border_res=sub.groupby(['split','border_group','model'],observed=True).agg(n_patients=('patient_id','nunique'),patient_dice_mean=('patient_dice','mean'),patient_dice_std=('patient_dice','std'),patient_recall_mean=('patient_recall','mean'),patient_precision_mean=('patient_precision','mean')).reset_index()
size_res.to_csv(TABLES/'results_by_lesion_size_quartile.csv',index=False);border_res.to_csv(TABLES/'results_by_crop_border.csv',index=False)
print(source[(source.split=='test')&(source.metric=='patient_dice')].to_string(index=False))

In [ ]:
order=MODELS
for split in ['test','external']:
    data=[seedavg[(seedavg.split==split)&(seedavg.model==m)].patient_dice.dropna().to_numpy() for m in order]
    fig,ax=plt.subplots(figsize=(8,5));ax.boxplot(data,tick_labels=[LABEL[m] for m in order],showmeans=True);ax.set_ylabel('Patient-level Dice');ax.set_title(f'Patient-level Dice — {split}');ax.grid(axis='y',alpha=.25);fig.tight_layout();fig.savefig(FIGURES/f'patient_dice_boxplot_{split}.png',dpi=220);plt.close(fig)

dice_ci=ci[ci.metric=='patient_dice']
for split in SPLITS:
    f=dice_ci[dice_ci.split==split].set_index('model').loc[order].reset_index();y=np.arange(len(f));means=f['mean'].to_numpy();err=np.vstack([means-f.ci95_lower.to_numpy(),f.ci95_upper.to_numpy()-means])
    fig,ax=plt.subplots(figsize=(8,4.5));ax.errorbar(means,y,xerr=err,fmt='o',capsize=4);ax.set_yticks(y);ax.set_yticklabels([LABEL[m] for m in f.model]);ax.set_xlabel('Mean patient-level Dice (95% bootstrap CI)');ax.set_title(f'Patient-level Dice confidence intervals — {split}');ax.grid(axis='x',alpha=.25);fig.tight_layout();fig.savefig(FIGURES/f'patient_dice_forest_{split}.png',dpi=220);plt.close(fig)
print('Figures:',len(list(FIGURES.glob('*.png'))))

In [ ]:
main=ci[ci.metric=='patient_dice'][['model','model_label','split','n_patients','mean','std','ci95_lower','ci95_upper']].copy()
main['mean_sd']=main.apply(lambda r:f"{r['mean']:.4f} ± {r['std']:.4f}",axis=1);main['ci95']=main.apply(lambda r:f"[{r['ci95_lower']:.4f}, {r['ci95_upper']:.4f}]",axis=1)
main.to_csv(TABLES/'main_patient_dice_table.csv',index=False)
ranking=main[main.split=='validation'].sort_values('mean',ascending=False)
swin=comp[(comp.metric=='patient_dice')&((comp.model_a=='swin_tiny_unet')|(comp.model_b=='swin_tiny_unet'))]
report=['# Track B MRI — Final patient-level statistical analysis','','All confidence intervals and paired comparisons use the patient as the statistical unit. The three independent seeds are averaged for each patient/model before inference statistics.','','## Validation-only ranking','',ranking.to_markdown(index=False),'','## Main Dice table','',main.sort_values(['split','mean'],ascending=[True,False]).to_markdown(index=False),'','## Paired Swin comparisons','',swin.to_markdown(index=False),'','Model selection must remain based on validation. Test and Duke are confirmatory.']
(WORK/'TRACKB_FINAL_STATISTICAL_REPORT.md').write_text('\n'.join(report),encoding='utf-8')
zip_path=Path('/kaggle/working/TRACKB_FINAL_STATISTICAL_ANALYSIS_LIGHT.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(WORK.parent))
print('Final ZIP:',zip_path);print(main.sort_values(['split','mean'],ascending=[True,False]).to_string(index=False))